# Prompt Robustness Sweep

**Goal:** Determine which prompt configurations generalize best across models and tasks for the LEVANTE benchmark.

## Experiment Design

We decompose each prompt into two orthogonal factors:

- **Task Framing (TF):** How the cognitive challenge is presented to the model (minimal, elimination, feature-grounded, step-by-step, etc.).
- **Output Format (OF):** How the model is instructed to respond (bare letter vs. JSON).

Each task has 4 TF variants × 2 OF variants = **8 prompt cells** per task.
CoT and structured tags were excluded from OF: CoT is 3-4× slower and yields low parse rates; tags showed 0% parse on 4B models.

### Models (5 models, 3 families)
| Model | Family | Size |
|-------|--------|------|
| Qwen2.5-VL | qwen | 0.8B |
| InternVL3.5 | internvl | 1B |
| SmolVLM2 | smolvlm | 2.2B |
| InternVL3.5 | internvl | 4B |
| Gemma3 | gemma | 4B |

### Tasks
| Task | Total trials | Subset (30%) | Options |
|------|-------------|-------------|---------|
| mental-rotation | 83 | 24 | 2 (A/B) |
| matrix-reasoning | 80 | 22 | 4 (A-D) |
| vocab | 170 | 49 | 4 (A-D) |
| trog | 99 | 28 | 4 (A-D) |
| theory-of-mind | 37 | 10 | 2-4 |
| egma-math | 179 | 51 | 4 (A-D) |

### Evaluation
- **Subset:** 30% stratified sample per task (preserves label distribution)
- **Selection criterion:** Maximin — maximize the worst-case accuracy across all models
- **Compute:** 5 models on 5× NVIDIA A40 GPUs in parallel (~2.5h total)

In [ ]:
from __future__ import annotations

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({"font.size": 11, "figure.dpi": 120})

df = pd.read_csv("../../results/sweep-meeting/sweep_results_merged.csv")
df["model_label"] = df["model"] + " " + df["model_size"].astype(str)
print(f"Loaded {len(df)} rows")
print(f"Models: {sorted(df['model_label'].unique())}")
print(f"Tasks: {sorted(df['task'].unique())}")
print(f"TF variants per task: {df.groupby('task')['task_framing'].nunique().to_dict()}")
print(f"OF variants: {sorted(df['output_format'].unique())}")

## 1. Global Model Performance

In [ ]:
summary = df.groupby("model_label").agg(
    mean_acc=("accuracy", "mean"),
    std_acc=("accuracy", "std"),
    min_acc=("accuracy", "min"),
    max_acc=("accuracy", "max"),
    mean_parse=("parse_rate", "mean"),
    min_parse=("parse_rate", "min"),
    n_cells=("accuracy", "count"),
).round(3).sort_values("mean_acc", ascending=False)
summary

## 2. Per-Task Heatmaps (Mean Accuracy Across Models)

In [ ]:
tasks = sorted(df["task"].unique())
n_tasks = len(tasks)
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, task in enumerate(tasks):
    ax = axes[idx]
    td = df[df["task"] == task]
    pivot = td.pivot_table(
        index="task_framing", columns="output_format",
        values="accuracy", aggfunc="mean"  # mean across models
    ).sort_index()
    
    im = ax.imshow(pivot.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, fontsize=9)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index, fontsize=9)
    ax.set_title(task, fontsize=12, fontweight="bold")
    
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            val = pivot.values[i, j]
            if not np.isnan(val):
                ax.text(j, i, f"{val:.0%}", ha="center", va="center",
                       fontsize=9, color="black" if 0.3 < val < 0.7 else "white")

fig.colorbar(im, ax=axes, shrink=0.4, label="Mean Accuracy")
fig.suptitle("Mean Accuracy Across 5 Models (TF × OF per Task)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../../results/sweep-meeting/heatmap_mean_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Min Accuracy Across Models (Maximin View)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, task in enumerate(tasks):
    ax = axes[idx]
    td = df[df["task"] == task]
    pivot = td.pivot_table(
        index="task_framing", columns="output_format",
        values="accuracy", aggfunc="min"  # worst-case across models
    ).sort_index()
    
    im = ax.imshow(pivot.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, fontsize=9)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index, fontsize=9)
    ax.set_title(task, fontsize=12, fontweight="bold")
    
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            val = pivot.values[i, j]
            if not np.isnan(val):
                ax.text(j, i, f"{val:.0%}", ha="center", va="center",
                       fontsize=9, color="black" if 0.3 < val < 0.7 else "white")

fig.colorbar(im, ax=axes, shrink=0.4, label="Min Accuracy (worst model)")
fig.suptitle("Maximin: Worst-Case Accuracy Across 5 Models (TF × OF per Task)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../../results/sweep-meeting/heatmap_min_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Output Format Comparison (OF1_bare vs. OF2_json)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy by OF
ax = axes[0]
of_data = df.groupby(["model_label", "output_format"]).agg(
    mean_acc=("accuracy", "mean")).reset_index()
of_pivot = of_data.pivot(index="model_label", columns="output_format", values="mean_acc")
of_pivot.plot(kind="bar", ax=ax, color=["#3498db", "#e74c3c"])
ax.set_ylabel("Mean Accuracy")
ax.set_title("Accuracy by Output Format", fontweight="bold")
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
ax.legend(title="OF")
ax.grid(axis="y", alpha=0.3)

# Parse rate by OF
ax = axes[1]
pr_data = df.groupby(["model_label", "output_format"]).agg(
    mean_parse=("parse_rate", "mean")).reset_index()
pr_pivot = pr_data.pivot(index="model_label", columns="output_format", values="mean_parse")
pr_pivot.plot(kind="bar", ax=ax, color=["#3498db", "#e74c3c"])
ax.set_ylabel("Mean Parse Rate")
ax.set_title("Parse Rate by Output Format", fontweight="bold")
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
ax.set_ylim(0.8, 1.02)
ax.legend(title="OF")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("../../results/sweep-meeting/of_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Model × Task Accuracy Profile

In [ ]:
model_order = ["smolvlm2 2.2B", "internvl35 1B", "qwen35 0.8B", "gemma3 4b-it", "internvl35 4B"]
task_order = ["mental-rotation", "matrix-reasoning", "trog", "vocab", "theory-of-mind", "egma-math"]

mt_pivot = df.pivot_table(
    index="model_label", columns="task",
    values="accuracy", aggfunc="mean"
).reindex(index=model_order, columns=task_order)

fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(mt_pivot.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(task_order)))
ax.set_xticklabels(task_order, rotation=30, ha="right", fontsize=10)
ax.set_yticks(range(len(model_order)))
ax.set_yticklabels(model_order, fontsize=10)

for i in range(len(model_order)):
    for j in range(len(task_order)):
        val = mt_pivot.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f"{val:.0%}", ha="center", va="center",
                   fontsize=10, fontweight="bold",
                   color="black" if 0.3 < val < 0.7 else "white")

fig.colorbar(im, label="Mean Accuracy")
ax.set_title("Mean Accuracy by Model × Task (averaged over TF × OF)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../../results/sweep-meeting/model_task_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Task Framing Effect (Accuracy Gain vs. TF1_minimal)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
colors = plt.cm.tab10(range(5))

for idx, task in enumerate(tasks):
    ax = axes[idx]
    td = df[df["task"] == task]
    
    # Mean accuracy per model per TF (average over OF)
    tf_perf = td.groupby(["model_label", "task_framing"])["accuracy"].mean().unstack(level=0)
    
    for i, model_label in enumerate(model_order):
        if model_label not in tf_perf.columns:
            continue
        vals = tf_perf[model_label].dropna()
        ax.plot(range(len(vals)), vals.values, "o-", color=colors[i],
                label=model_label, markersize=5, linewidth=1.5)
    
    ax.set_xticks(range(len(vals)))
    ax.set_xticklabels(vals.index, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("Accuracy")
    ax.set_title(task, fontsize=12, fontweight="bold")
    ax.grid(alpha=0.3)
    if idx == 0:
        ax.legend(fontsize=7, loc="best")

fig.suptitle("Task Framing Effect per Model", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../../results/sweep-meeting/tf_effect.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Response Bias Analysis

In [ ]:
bias_summary = df.groupby("model_label").agg(
    mean_bias=("bias_ratio", "mean"),
    max_bias=("bias_ratio", "max"),
    mean_parse=("parse_rate", "mean"),
).round(3).sort_values("mean_bias", ascending=False)

print("Response Bias Summary (higher = more biased toward one label):")
print(bias_summary.to_string())
print()

# Worst bias cases
high_bias = df[df["bias_ratio"] > 0.7][["model_label", "task", "task_framing", 
                                          "output_format", "bias_ratio", "dominant_label", "accuracy"]]
print(f"\nHigh-bias cells (bias_ratio > 0.7): {len(high_bias)}")
if len(high_bias) > 0:
    print(high_bias.sort_values("bias_ratio", ascending=False).head(15).to_string(index=False))

## 8. Maximin Recommendations

In [ ]:
print("Best TF × OF per Task (maximin criterion: maximize worst-case model accuracy)")
print("=" * 90)

recommendations = []
for task in tasks:
    td = df[df["task"] == task]
    pivot = td.pivot_table(
        index=["task_framing", "output_format"],
        columns="model_label", values="accuracy"
    )
    pivot["min_acc"] = pivot.min(axis=1)
    pivot["mean_acc"] = pivot.mean(axis=1)
    pivot["max_acc"] = pivot.max(axis=1)
    best = pivot.sort_values("min_acc", ascending=False).head(3)
    
    print(f"\n{task}:")
    for (tf, of), row in best.iterrows():
        marker = "★" if row["min_acc"] == best["min_acc"].max() else " "
        print(f"  {marker} {tf} × {of}: min={row['min_acc']:.1%}  "
              f"mean={row['mean_acc']:.1%}  max={row['max_acc']:.1%}")
        recommendations.append({
            "task": task, "task_framing": tf, "output_format": of,
            "min_acc": round(row["min_acc"], 3),
            "mean_acc": round(row["mean_acc"], 3),
            "max_acc": round(row["max_acc"], 3),
            "rank": 1 if row["min_acc"] == best["min_acc"].max() else 2
        })

rec_df = pd.DataFrame(recommendations)
rec_df.to_csv("../../results/sweep-meeting/recommendations.csv", index=False)
print(f"\nSaved recommendations to results/sweep-meeting/recommendations.csv")

## 9. Conclusions

### Output Format
- **OF1_bare and OF2_json perform similarly** on accuracy (46.9% vs. 45.7% mean).
- OF1_bare has slightly higher parse rates overall (98.1% vs. 96.8%).
- **Recommendation:** Use OF2_json as default — it provides structured output for automated analysis with comparable accuracy, and 100% parse on most model/task combinations.

### Task Framing
- **TF1_minimal consistently ranks in the top-2** across most tasks (maximin).
- Elaborate framings (elimination, step-by-step) rarely help and can hurt smaller models.
- Some task-specific framings show large gains for specific models but hurt others (e.g., TF2_naming boosts InternVL 4B on vocab to 100% but doesn't help smaller models as much).

### Global Maximin Recommendation
For a **single universal prompt** to run across all models:
1. **TF1_minimal × OF2_json** — best worst-case floor in 4/6 tasks
2. **TF1_minimal × OF1_bare** — close second, slightly higher ceiling in some tasks

### Per-Family Optimization Possible
- The 4B models (InternVL 4B, Gemma3) benefit more from enriched framings
- Sub-1B models perform best with minimal framing (less prompt confusion)
- If running per-family prompts is feasible, task-specific TF variants can yield 5-15% gains